# Fuzzy-Logic Standard-Attention Baseline vs. Shared-Term Contrastive Loss

This notebook runs a controlled lambda sweep on a Kaggle or Colab GPU. Every run uses the official repository's **standard softmax attention** (`attention_norm=softmax`, `target_network=default`), not HyLA attention.

The effective official model and optimization settings are retained: sequence length 16, two layers, embedding dimension 128, eight heads, QK/value dimensions 16, MLP dimension 256, block dropout 0, and attention dropout 0.1. Dataset size, epoch count, batch size, and lambda are explicit experiment parameters. Training displays only a sample-count tqdm bar; complete ID/test/OOD metrics are reported after each full epoch. Each model runs in a separate subprocess and releases its GPU context before the next model starts.

## 1. Select a GPU runtime

On Kaggle, open **Session options** and set **Accelerator → GPU**. On Colab, choose **Runtime → Change runtime type → GPU**. Starting or changing the accelerator restarts the session, so do this before running any cells.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU detected. Select a GPU runtime and reconnect.")
subprocess.run(["nvidia-smi"], check=True)

## 2. Upload and extract the standalone experiment

**Kaggle:** In the notebook editor's right sidebar, open **Input**, click **Add Input** (or **Upload**), upload `fuzzy_logic_attention_contrastive.zip` as a private dataset, and attach it to this notebook. Kaggle mounts it under `/kaggle/input`; the next cell finds it automatically and extracts the code into writable `/kaggle/working`.

**Colab:** Upload the same ZIP into `/content`. If it is absent, the next cell opens the Colab upload dialog.

Create the archive locally from the parent `experiments` directory with:

```bash
zip -r fuzzy_logic_attention_contrastive.zip fuzzy_logic_attention_contrastive -x '*/__pycache__/*' '*/logs/*' '*/wandb/*'
```

In [ ]:
from pathlib import Path
import shutil
import zipfile

ON_KAGGLE = Path("/kaggle/input").exists()
PLATFORM_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
EXPERIMENT_ROOT = PLATFORM_ROOT / "fuzzy_logic_attention_contrastive"

if EXPERIMENT_ROOT.exists():
    shutil.rmtree(EXPERIMENT_ROOT)

if ON_KAGGLE:
    input_root = Path("/kaggle/input")
    exact_zips = list(input_root.rglob("fuzzy_logic_attention_contrastive.zip"))
    zip_candidates = exact_zips or list(input_root.rglob("*.zip"))
    if zip_candidates:
        ZIP_PATH = zip_candidates[0]
        print("Using attached Kaggle ZIP:", ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall(PLATFORM_ROOT)
    else:
        # Kaggle may expose an uploaded archive as already-extracted files.
        source_candidates = [
            path.parent for path in input_root.rglob("run.py")
            if (path.parent / "configs/logic.py").exists()
        ]
        if len(source_candidates) != 1:
            raise FileNotFoundError(
                "Attach the experiment ZIP using Kaggle Input > Add Input/Upload. "
                f"Found ZIPs={zip_candidates}, code roots={source_candidates}"
            )
        shutil.copytree(source_candidates[0], EXPERIMENT_ROOT)
else:
    from google.colab import files
    ZIP_PATH = PLATFORM_ROOT / "fuzzy_logic_attention_contrastive.zip"
    if not ZIP_PATH.exists():
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError("Upload exactly one experiment ZIP file.")
        uploaded_path = Path(zip_names[0])
        if uploaded_path.resolve() != ZIP_PATH.resolve():
            shutil.move(uploaded_path, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(PLATFORM_ROOT)

required = ["run.py", "requirements-colab.txt", "configs/logic.py", "hyla/loss.py"]
missing = [name for name in required if not (EXPERIMENT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f"ZIP layout is incorrect; missing: {missing}")
print("Experiment extracted to:", EXPERIMENT_ROOT)

## 3. Install the lightweight accelerator-compatible dependencies

The platform's accelerator-compatible JAX installation is retained. Kaggle uses `requirements-kaggle.txt` with `flax==0.11.2`, which is compatible with Kaggle's preinstalled JAX 0.7.x stack and does not upgrade JAX, jaxlib, or its CUDA plugin. Colab uses `requirements-colab.txt`. On Kaggle, enable **Internet** if pip needs to download a missing package. If an earlier install produced a JAX/plugin mismatch, restart the Kaggle session before running this cell.

In [ ]:
import os
import subprocess
import sys
os.chdir(EXPERIMENT_ROOT)
print("Working directory:", Path.cwd())
requirements_file = "requirements-kaggle.txt" if ON_KAGGLE else "requirements-colab.txt"
print("Installing:", requirements_file)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file],
    check=True,
)

In [ ]:
import subprocess
import sys

# Check JAX in a short-lived subprocess so the notebook process does not keep a CUDA context.
device_check = (
    "import importlib.metadata as md; import jax, jax.numpy as jnp, jaxlib, flax; "
    "from flax.core.tracers import current_trace; "
    "current_trace(); "
    "print('JAX:', jax.__version__); "
    "print('jaxlib:', jaxlib.__version__); "
    "print('Flax:', flax.__version__); "
    "plugin = next((md.version(n) for n in ['jax-cuda12-plugin', 'jax_cuda12_plugin'] if n in {d.metadata['Name'] for d in md.distributions()}), None); "
    "print('CUDA plugin:', plugin); "
    "print('Backend:', jax.default_backend()); "
    "print('Devices:', jax.devices()); "
    "assert jax.default_backend() == 'gpu', 'JAX is not using the notebook GPU accelerator'; "
    "assert plugin is None or plugin == jaxlib.__version__, f'CUDA plugin {plugin} != jaxlib {jaxlib.__version__}'; "
    "result = jax.jit(lambda x: x * 2)(jnp.arange(4)).block_until_ready(); "
    "print('GPU JIT smoke test:', result)"
)
subprocess.run([sys.executable, "-c", device_check], check=True)

## 4. Use platform-local writable result storage

On Kaggle, outputs are written under `/kaggle/working/fuzzy_logic_attention_results`; use **Save Version** before the session ends so Kaggle preserves them as notebook output. On Colab, they are written under `/content/fuzzy_logic_attention_results`. Nothing is written to Google Drive.

In [ ]:
from pathlib import Path

EXPERIMENT_ROOT = PLATFORM_ROOT / "fuzzy_logic_attention_contrastive"
RESULTS_ROOT = PLATFORM_ROOT / "fuzzy_logic_attention_results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be saved under:", RESULTS_ROOT)

## 5. Validate the controlled reference configuration

This cell verifies the untouched four-variable/two-term reference configuration: 6,400,000 generated training samples, one epoch, and 50,000 optimizer steps at batch size 128. Later cells deliberately vary dataset size and batch size while retaining the architecture and optimizer settings. Batch sizes 32, 64, and 128 fit this small two-layer Transformer on a T4.

In [ ]:
import sys

sys.path.insert(0, str(EXPERIMENT_ROOT))
from configs.logic import get_config

cfg = get_config("logic_4var_2term;transformer")
expected = {
    "batch_size": 128,
    "seq_len": 16,
    "num_layers": 2,
    "emb_dim": 128,
    "num_heads": 8,
    "qk_dim": 16,
    "v_dim": 16,
    "mlp_dim": 256,
    "dropout_rate": 0.0,
    "attention_dropout_rate": 0.1,
    "sequence_mixer": "softmax_attention",
    "learning_rate": 0.001,
    "weight_decay": 0.1,
    "warmup_steps": 100,
    "training_steps": 50_000,
    "dataset_size": 6_400_000,
    "num_epochs": 1,
    "temperature": 1.0,
}
actual = {
    "batch_size": cfg.batch_size,
    "seq_len": cfg.data.seq_len,
    **{
        key: cfg.model[key]
        for key in expected
        if key not in {"batch_size", "seq_len", "learning_rate", "weight_decay", "warmup_steps", "training_steps", "dataset_size", "num_epochs", "temperature"}
    },
    "learning_rate": cfg.lr,
    "weight_decay": cfg.weight_decay,
    "warmup_steps": cfg.warmup_steps,
    "training_steps": cfg.data.num_train // cfg.batch_size,
    "dataset_size": cfg.data.num_train,
    "num_epochs": cfg.num_epochs,
    "temperature": cfg.temperature,
}
assert actual == expected, f"Configuration mismatch: {actual}"
for key, value in actual.items():
    print(f"{key:26s} = {value}")

task_specs = {
    "logic_3var_2term": (3, 2),
    "logic_4var_2term": (4, 2),
    "logic_4var_3term": (4, 3),
    "logic_5var_2term": (5, 2),
}
for task_name, (num_variables, num_terms) in task_specs.items():
    task_cfg = get_config(f"{task_name};transformer")
    assert (task_cfg.data.num_variables, task_cfg.data.num_terms) == (num_variables, num_terms)
    assert all(task_cfg.data.num_valid % batch == 0 for batch in (32, 64, 128))
print("Validated all four task configurations and evaluation batch divisibility.")

## 6. Define the isolated runner and automatic GPU cleanup

JAX normally preallocates GPU memory. The child processes below disable preallocation. More importantly, each model owns a separate process; process termination releases its CUDA context before the next run. Cleanup happens in a `finally` block even if training fails.

In [ ]:
import gc
import json
import os
import subprocess
import sys
import time

def report_gpu_memory():
    subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
        check=False,
    )

def release_after_model():
    # The training subprocess has already exited, which releases its CUDA context.
    collected = gc.collect()
    time.sleep(2)
    print(f"Python garbage collector reclaimed {collected} objects.")
    print("GPU processes after cleanup:")
    report_gpu_memory()

def load_complete_metrics(metrics_path, dataset_size, num_epochs):
    try:
        metrics = json.loads(metrics_path.read_text())
    except (OSError, json.JSONDecodeError):
        return None
    complete = (
        int(metrics.get("dataset_size", -1)) == int(dataset_size)
        and int(metrics.get("num_epochs", -1)) == int(num_epochs)
        and metrics_path.with_name("state.pkl").exists()
        and any(metrics_path.parent.glob("latent_dataset_step_*.pkl"))
    )
    return metrics if complete else None

def run_attention_analysis(log_dir, seed):
    analysis_path = log_dir / "attention_analysis.json"
    if analysis_path.exists():
        print("Attention analysis already complete:", analysis_path)
        return
    command = [
        sys.executable, "analyze_attention.py",
        f"--run_dir={log_dir}",
        f"--max_samples={ATTENTION_ANALYSIS_SAMPLES}",
        f"--seed={seed}",
    ]
    print("Running paper-style attention analysis:", " ".join(command))
    subprocess.run(command, cwd=EXPERIMENT_ROOT, check=True)

def run_model(name, task_name, dataset_size, num_epochs, batch_size, lambda_contrastive, seed, temperature=1.0):
    run_root = RESULTS_ROOT / name
    run_root.mkdir(parents=True, exist_ok=True)
    for metrics_path in sorted(run_root.rglob("final_metrics.json"), reverse=True):
        if load_complete_metrics(metrics_path, dataset_size, num_epochs) is not None:
            run_attention_analysis(metrics_path.parent, seed)
            print(f"Skipping completed run {name}: {metrics_path}")
            return

    command = [
        sys.executable,
        "run.py",
        f"--config=configs/logic.py:{task_name};transformer",
        f"--config.lambda_contrastive={lambda_contrastive}",
        f"--config.temperature={temperature}",
        f"--config.seed={seed}",
        f"--config.batch_size={batch_size}",
        f"--config.data.num_train={dataset_size}",
        f"--config.num_epochs={num_epochs}",
        "--config.data.seq_len=16",
        "--config.lr=0.001",
        "--config.weight_decay=0.1",
        "--log_level=1",
        "--logger=standard",
        f"--workdir={run_root}",
    ]
    environment = os.environ.copy()
    environment.update({
        "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
        "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.85",
        "PYTHONUNBUFFERED": "1",
    })

    print(
        f"Starting {name}: task={task_name}, samples/epoch={dataset_size:,}, "
        f"epochs={num_epochs}, batch={batch_size}, lambda={lambda_contrastive}, "
        f"seed={seed}, temperature={temperature}"
    )
    print("Command:", " ".join(command))
    try:
        completed_process = subprocess.run(
            command,
            cwd=EXPERIMENT_ROOT,
            env=environment,
            check=False,
        )
        if completed_process.returncode != 0:
            raise RuntimeError(
                f"{name} exited with status {completed_process.returncode}. "
                "See the traceback immediately above."
            )
        completed = [
            path for path in sorted(run_root.rglob("final_metrics.json"), reverse=True)
            if load_complete_metrics(path, dataset_size, num_epochs) is not None
        ]
        if not completed:
            raise RuntimeError(f"{name} finished without complete final metrics.")
        run_attention_analysis(completed[0].parent, seed)
    finally:
        release_after_model()

## 7. Configure dataset-size, epoch, task, batch-size, lambda, and seed variations

Dataset sizes are 128,000 (2% of reference), 640,000 (10%), and the paper-scale 6,400,000 samples. `NUM_EPOCHS` controls complete passes over that deterministic generated dataset. With a fixed dataset size, changing batch size changes optimizer-step count but not the number of examples seen. Lambda 0 is the MSE-only baseline; temperature stays fixed at 1.0. Every epoch shows processed samples/total samples and performs one full ID/test/OOD evaluation only at its end. The default one-seed matrix contains 144 runs, divided among 12 independently resumable cells below.

In [ ]:
TASK_NAMES = [
    "logic_3var_2term",
    "logic_4var_2term",
    "logic_4var_3term",
    "logic_5var_2term",
]
BATCH_SIZES = [32, 64, 128]
LAMBDA_VALUES = [0.0, 0.01, 0.05, 0.1]
DATASET_SIZES = [128_000, 640_000, 6_400_000]
NUM_EPOCHS = 1  # Increase this to make multiple complete passes over each dataset.
SEEDS = [0]  # Change to [0, 1, 2] for the full three-seed study.
TEMPERATURE = 1.0
ATTENTION_ANALYSIS_SAMPLES = 2_000

assert all(size % batch == 0 for size in DATASET_SIZES for batch in BATCH_SIZES)
num_runs = len(TASK_NAMES) * len(DATASET_SIZES) * len(BATCH_SIZES) * len(LAMBDA_VALUES) * len(SEEDS)
print(f"Configured {num_runs} runs across 12 resumable variation cells.")
print("Epochs per run:", NUM_EPOCHS)
for size in DATASET_SIZES:
    print(f"{size:,} samples/epoch -> optimizer steps by batch:", {b: size // b for b in BATCH_SIZES})

def run_task_variation(task_name, dataset_size):
    for batch_size in BATCH_SIZES:
        for lambda_value in LAMBDA_VALUES:
            lambda_tag = f"{lambda_value:g}".replace(".", "p")
            for seed in SEEDS:
                run_model(
                    name=(
                        f"samples_{dataset_size}/epochs_{NUM_EPOCHS}/{task_name}/"
                        f"batch_{batch_size}/lambda_{lambda_tag}/seed_{seed}"
                    ),
                    task_name=task_name, dataset_size=dataset_size,
                    num_epochs=NUM_EPOCHS, batch_size=batch_size,
                    lambda_contrastive=lambda_value, seed=seed,
                    temperature=TEMPERATURE,
                )

## 8. Run variations in separate, resumable cells

Each cell covers one task and one dataset size (12 batch/lambda models). If a model fails, fix the issue and rerun only that cell: completed models and completed attention analyses are skipped. Run only the variations you need.

In [ ]:
# 3 variables, 2 terms - 128,000 samples/epoch
run_task_variation("logic_3var_2term", 128_000)

In [ ]:
# 3 variables, 2 terms - 640,000 samples/epoch
run_task_variation("logic_3var_2term", 640_000)

In [ ]:
# 3 variables, 2 terms - reference 6,400,000 samples/epoch
run_task_variation("logic_3var_2term", 6_400_000)

In [ ]:
# 4 variables, 2 terms - 128,000 samples/epoch
run_task_variation("logic_4var_2term", 128_000)

In [ ]:
# 4 variables, 2 terms - 640,000 samples/epoch
run_task_variation("logic_4var_2term", 640_000)

In [ ]:
# 4 variables, 2 terms - reference 6,400,000 samples/epoch
run_task_variation("logic_4var_2term", 6_400_000)

In [ ]:
# 4 variables, 3 terms - 128,000 samples/epoch
run_task_variation("logic_4var_3term", 128_000)

In [ ]:
# 4 variables, 3 terms - 640,000 samples/epoch
run_task_variation("logic_4var_3term", 640_000)

In [ ]:
# 4 variables, 3 terms - reference 6,400,000 samples/epoch
run_task_variation("logic_4var_3term", 6_400_000)

In [ ]:
# 5 variables, 2 terms - 128,000 samples/epoch
run_task_variation("logic_5var_2term", 128_000)

In [ ]:
# 5 variables, 2 terms - 640,000 samples/epoch
run_task_variation("logic_5var_2term", 640_000)

In [ ]:
# 5 variables, 2 terms - reference 6,400,000 samples/epoch
run_task_variation("logic_5var_2term", 6_400_000)

## 9. Export tables and comparison plots

Each run writes one complete ID/test/OOD evaluation per epoch to `evaluation_history.csv`, plus `final_metrics.json`. This cell aggregates completed runs by dataset size, epoch count, task, batch size, lambda, and seed. It exports performance plots for every dataset size and preserves each model's t-SNE and term-decoding analysis. `test_r2` is the main compositional OOD metric.

In [ ]:
import json
import shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, display

records = []
for metrics_path in sorted(RESULTS_ROOT.rglob("final_metrics.json")):
    try:
        record = json.loads(metrics_path.read_text())
    except json.JSONDecodeError:
        print("Ignoring incomplete metrics file:", metrics_path)
        continue
    if not {"dataset_size", "num_epochs"}.issubset(record):
        print("Ignoring result from the older step-budget notebook:", metrics_path)
        continue
    if not metrics_path.with_name("attention_analysis.json").exists():
        print("Ignoring run without completed attention analysis:", metrics_path)
        continue
    record["metrics_path"] = str(metrics_path)
    records.append(record)
if not records:
    raise RuntimeError("No completed final_metrics.json files were found.")

runs = pd.DataFrame(records).sort_values(
    ["dataset_size", "num_epochs", "task_name", "batch_size", "lambda_contrastive", "seed"]
)
runs_path = RESULTS_ROOT / "experiment_runs.csv"
runs.to_csv(runs_path, index=False)

metric_columns = ["id_mse", "id_r2", "test_mse", "test_r2", "ood_mse", "ood_r2"]
positive_columns = [f"{split}_avg_num_positives" for split in ["id", "test", "ood"]]
analysis_columns = ["attention_mean_macro_f1", "attention_mean_accuracy"]
for column in analysis_columns:
    if column not in runs:
        runs[column] = float("nan")
group_columns = ["dataset_size", "num_epochs", "task_name", "batch_size", "lambda_contrastive"]
aggregate = runs.groupby(group_columns)[metric_columns + positive_columns + analysis_columns].agg(["mean", "std", "count"])
aggregate.columns = [f"{metric}_{stat}" for metric, stat in aggregate.columns]
aggregate = aggregate.reset_index()
aggregate_path = RESULTS_ROOT / "experiment_aggregate.csv"
aggregate.to_csv(aggregate_path, index=False)

history_frames = []
for metrics_path_string in runs["metrics_path"]:
    metrics_path = Path(metrics_path_string)
    metadata = json.loads(metrics_path.read_text())
    history_path = metrics_path.with_name("evaluation_history.csv")
    if history_path.exists():
        history = pd.read_csv(history_path)
        for key in ["dataset_size", "num_epochs", "task_name", "batch_size", "lambda_contrastive", "seed"]:
            history[key] = metadata[key]
        history_frames.append(history)
checkpoint_history = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
history_export_path = RESULTS_ROOT / "checkpoint_history.csv"
checkpoint_history.to_csv(history_export_path, index=False)

display(runs[["dataset_size", "num_epochs", "task_name", "batch_size", "lambda_contrastive", "seed", *metric_columns, *positive_columns, *analysis_columns]])
display(aggregate)

def plot_grid(metric, ylabel, filename_prefix):
    outputs = []
    for dataset_size in sorted(runs["dataset_size"].unique()):
        size_runs = runs[runs["dataset_size"] == dataset_size]
        task_names = sorted(size_runs["task_name"].unique())
        splits = ["id", "test", "ood"]
        fig, axes = plt.subplots(
            len(task_names), len(splits), figsize=(15, 4.2 * len(task_names)),
            squeeze=False, sharex=True,
        )
        for row, task_name in enumerate(task_names):
            task_runs = size_runs[size_runs["task_name"] == task_name]
            for column, split in enumerate(splits):
                ax = axes[row, column]
                key = f"{split}_{metric}"
                for batch_size in sorted(task_runs["batch_size"].unique()):
                    subset = task_runs[task_runs["batch_size"] == batch_size]
                    summary = subset.groupby("lambda_contrastive")[key].agg(["mean", "std"])
                    ax.errorbar(
                        summary.index, summary["mean"], yerr=summary["std"].fillna(0),
                        marker="o", capsize=4, label=f"batch={batch_size}",
                    )
                ax.set_title(f"{task_name} - {split}")
                ax.set_xlabel("Contrastive loss weight lambda")
                ax.set_ylabel(ylabel)
                ax.grid(alpha=0.25)
                if row == 0 and column == 0:
                    ax.legend()
        fig.suptitle(f"{dataset_size:,} training samples per epoch", y=1.002)
        fig.tight_layout()
        output = RESULTS_ROOT / f"{filename_prefix}_samples_{dataset_size}.png"
        fig.savefig(output, dpi=180, bbox_inches="tight")
        plt.show()
        outputs.append(output)
    return outputs

r2_plots = plot_grid("r2", "R²", "experiment_r2")
mse_plots = plot_grid("mse", "MSE", "experiment_mse")

def plot_dataset_scaling():
    task_names = sorted(runs["task_name"].unique())
    batch_sizes = sorted(runs["batch_size"].unique())
    fig, axes = plt.subplots(
        len(task_names), len(batch_sizes),
        figsize=(5 * len(batch_sizes), 4.2 * len(task_names)), squeeze=False,
    )
    for row, task_name in enumerate(task_names):
        for column, batch_size in enumerate(batch_sizes):
            ax = axes[row, column]
            subset = runs[(runs["task_name"] == task_name) & (runs["batch_size"] == batch_size)]
            for lambda_value in sorted(subset["lambda_contrastive"].unique()):
                curve = subset[subset["lambda_contrastive"] == lambda_value].groupby("dataset_size")["test_r2"].agg(["mean", "std"])
                ax.errorbar(
                    curve.index, curve["mean"], yerr=curve["std"].fillna(0),
                    marker="o", capsize=4, label=f"lambda={lambda_value:g}",
                )
            ax.set_xscale("log")
            ax.set_title(f"{task_name} - batch={batch_size}")
            ax.set_xlabel("Training samples per epoch (log scale)")
            ax.set_ylabel("Compositional test R²")
            ax.grid(alpha=0.25)
            if row == 0 and column == 0:
                ax.legend()
    fig.tight_layout()
    output = RESULTS_ROOT / "experiment_dataset_scaling_test_r2.png"
    fig.savefig(output, dpi=180, bbox_inches="tight")
    plt.show()
    return output

dataset_scaling_plot = plot_dataset_scaling()

def plot_positive_counts():
    task_names = sorted(runs["task_name"].unique())
    splits = ["id", "test", "ood"]
    fig, axes = plt.subplots(
        len(task_names), len(splits), figsize=(15, 4.2 * len(task_names)),
        squeeze=False, sharex=True,
    )
    for row, task_name in enumerate(task_names):
        task_runs = runs[runs["task_name"] == task_name]
        for column, split in enumerate(splits):
            ax = axes[row, column]
            key = f"{split}_avg_num_positives"
            summary = task_runs.groupby("batch_size")[key].agg(["mean", "std"])
            ax.errorbar(
                summary.index, summary["mean"], yerr=summary["std"].fillna(0),
                marker="o", capsize=4,
            )
            ax.set_title(f"{task_name} — {split}")
            ax.set_xlabel("Batch size")
            ax.set_ylabel("Average positive partners / anchor")
            ax.grid(alpha=0.25)
    fig.tight_layout()
    output = RESULTS_ROOT / "experiment_positive_counts.png"
    fig.savefig(output, dpi=180, bbox_inches="tight")
    plt.show()
    return output

positive_plot = plot_positive_counts()

export_dir = RESULTS_ROOT / "export"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir()
for source in [runs_path, aggregate_path, history_export_path, positive_plot, dataset_scaling_plot, *r2_plots, *mse_plots]:
    shutil.copy2(source, export_dir / source.name)
analysis_names = ["attention_analysis.json", "attention_term_f1.csv", "attention_term_f1.png", "attention_tsne.png"]
for metrics_path_string in runs["metrics_path"]:
    log_dir = Path(metrics_path_string).parent
    relative = log_dir.relative_to(RESULTS_ROOT)
    destination = export_dir / "attention_analysis" / relative
    destination.mkdir(parents=True, exist_ok=True)
    for name in analysis_names:
        source = log_dir / name
        if source.exists():
            shutil.copy2(source, destination / name)
archive_path = Path(shutil.make_archive(
    str(PLATFORM_ROOT / "fuzzy_logic_attention_results_export"), "zip", root_dir=export_dir
))
print("Saved export files under:", export_dir)
if ON_KAGGLE:
    print("Kaggle export ready. Download it from the Output pane or use this link:")
    display(FileLink(str(archive_path)))
    print("Use Save Version before ending the session to preserve /kaggle/working outputs.")
else:
    from google.colab import files
    print("Downloading:", archive_path)
    files.download(str(archive_path))